In [0]:
bronze_rows = [
    (1, 1, 2, 50.00, "COMPLETED"),
    (2, 2, 1, 40.00, "COMPLETED"),
    (3, 4, 3, 36.00, "COMPLETED"),
]
bronze_cols = ["sales_key", "product_key", "quantity", "line_amount", "status"]

In [0]:
update_rows = [
    (2, 2, 1, 40.00, "CANCELLED"),  
    (4, 3, 1, 45.00, "COMPLETED"),   
]


In [0]:
from pyspark.sql import functions 

In [0]:
path = "/Volumes/workspace/default/de_lab/sales_bronze"
df = spark.createDataFrame(bronze_rows, bronze_cols)
updated_df = spark.createDataFrame(update_rows, bronze_cols)
df.write.format("delta").mode("overwrite").save(path)
updated_x = updated_df.createOrReplaceTempView("updates")

In [0]:
spark.sql(f"""
MERGE INTO delta.`{path}` AS t
USING updates AS s
ON t.sales_key = s.sales_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

In [0]:
spark.read.format("delta").load(path).orderBy("sales_key").show()
spark.sql(f"DESCRIBE HISTORY delta.`{path}`").select("version", "operation").show()
spark.read.format("delta").option("versionAsOf", 0).load(path).show()